# COCO-small Pipeline (Colab, Drive Direct)

??? ?????? ???????? ???????? ? ????????? ? Google Drive ? ?? ???????? `coco_small_yolo` ? `/content/datasets`.
???????? ??? ???????? ?????? (??? ?????? ?????????? ????????? ?????).


In [ ]:
# Step 0: install dependencies in Colab runtime.
%pip install -q ultralytics pycocotools albumentations pyyaml tqdm


In [ ]:
# Step 1: mount Google Drive and define dataset/experiment paths.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive')
COCO_ZIPS_DIR = DRIVE_ROOT / 'datasets' / 'coco2017_zips'
COCO_RAW_DIR = DRIVE_ROOT / 'datasets' / 'coco2017_raw'
COCO_SMALL_YOLO_DIR = DRIVE_ROOT / 'datasets' / 'coco_small_yolo'

COCO_EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments' / 'small_objects_auto_aug' / 'coco_small'
COCO_EXPERIMENT_RUNS = COCO_EXPERIMENT_ROOT / 'runs'
COCO_EXPERIMENT_ARTIFACTS = COCO_EXPERIMENT_ROOT / 'artifacts'

for path in [
    COCO_ZIPS_DIR,
    COCO_RAW_DIR,
    COCO_SMALL_YOLO_DIR,
    COCO_EXPERIMENT_RUNS,
    COCO_EXPERIMENT_ARTIFACTS,
]:
    path.mkdir(parents=True, exist_ok=True)

print('COCO zips dir:', COCO_ZIPS_DIR)
print('COCO raw dir:', COCO_RAW_DIR)
print('COCO small yolo dir:', COCO_SMALL_YOLO_DIR)
print('COCO experiment root:', COCO_EXPERIMENT_ROOT)


In [ ]:
# Step 2: clone/open project repository in runtime.
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path('/content/small_objects_auto_aug')
REPO_URL = 'https://github.com/s44w/small_objects_auto_aug.git'

if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Using PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# Step 3: download official COCO 2017 archives and ensure they are saved to Google Drive.
from src.data.coco_small_manager import COCO_ARCHIVES_2017
import shutil
import subprocess
import urllib.request
import zipfile

FORCE_DOWNLOAD_COCO = False
REPAIR_CORRUPTED_ARCHIVES = True
TMP_DOWNLOAD_DIR = Path('/content/coco_tmp_downloads')
TMP_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)


def _is_valid_zip(path: Path) -> bool:
    if not path.exists() or path.stat().st_size <= 0:
        return False
    if not zipfile.is_zipfile(path):
        return False
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            return zf.testzip() is None
    except Exception:
        return False


def _download_with_fallback(url: str, temp_target: Path) -> str:
    candidates = []
    if url.startswith('http://'):
        candidates += [url.replace('http://', 'https://', 1), url]
    elif url.startswith('https://'):
        candidates += [url, url.replace('https://', 'http://', 1)]
    else:
        candidates += [url]

    ordered = []
    seen = set()
    for u in candidates:
        if u not in seen:
            seen.add(u)
            ordered.append(u)

    errors = []
    for u in ordered:
        try:
            if temp_target.exists():
                temp_target.unlink()
            print(f'[TRY] urllib: {u}')
            urllib.request.urlretrieve(u, filename=str(temp_target))
            if _is_valid_zip(temp_target):
                return f'urllib:{u}'
            errors.append(f'invalid zip via urllib: {u}')
        except Exception as exc:
            errors.append(f'urllib failed: {u} :: {exc}')
        finally:
            if temp_target.exists() and not _is_valid_zip(temp_target):
                temp_target.unlink(missing_ok=True)

    https_url = ordered[0]
    try:
        if temp_target.exists():
            temp_target.unlink()
        print(f'[TRY] wget --no-check-certificate: {https_url}')
        subprocess.check_call(['wget', '--no-check-certificate', '-O', str(temp_target), https_url])
        if _is_valid_zip(temp_target):
            return f'wget:{https_url}'
        errors.append(f'invalid zip via wget: {https_url}')
    except Exception as exc:
        errors.append(f'wget failed: {https_url} :: {exc}')

    if temp_target.exists() and not _is_valid_zip(temp_target):
        temp_target.unlink(missing_ok=True)
    raise RuntimeError('Failed to download valid COCO archive. Attempts: ' + ' | '.join(errors[-6:]))


archive_paths = {name: (COCO_ZIPS_DIR / name) for name in COCO_ARCHIVES_2017.keys()}
missing = [name for name, path in archive_paths.items() if not path.exists()]
corrupted = [name for name, path in archive_paths.items() if path.exists() and not _is_valid_zip(path)]

need_download = set(missing)
if REPAIR_CORRUPTED_ARCHIVES:
    need_download.update(corrupted)
if FORCE_DOWNLOAD_COCO:
    need_download = set(archive_paths.keys())

if need_download:
    print('[INFO] Archives to download/re-download:', ', '.join(sorted(need_download)))
    for name in sorted(need_download):
        temp_target = TMP_DOWNLOAD_DIR / name
        drive_target = archive_paths[name]
        source = _download_with_fallback(COCO_ARCHIVES_2017[name], temp_target)
        shutil.copy2(temp_target, drive_target)
        if not _is_valid_zip(drive_target):
            raise RuntimeError(f'Archive was downloaded but not saved correctly to Google Drive: {drive_target}')
        print(f'[OK] {name} downloaded via {source} and saved to Drive: {drive_target}')
else:
    print('[SKIP] All COCO archives are already present and valid on Google Drive.')

invalid_after = [name for name, path in archive_paths.items() if not _is_valid_zip(path)]
if invalid_after:
    raise RuntimeError('Invalid archives remain on Google Drive: ' + ', '.join(invalid_after))

for name, path in archive_paths.items():
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    print(f'[OK] {name}: {path} ({size_mb:.1f} MB)')


In [ ]:
# Step 4: extract COCO archives in Google Drive.
from src.data.coco_small_manager import COCO_ARCHIVES_2017, extract_coco_2017_archives
import zipfile

FORCE_EXTRACT_COCO = False


def _is_valid_zip(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0 and zipfile.is_zipfile(path)


archive_paths = {name: (COCO_ZIPS_DIR / name) for name in COCO_ARCHIVES_2017.keys()}
invalid = [name for name, path in archive_paths.items() if not _is_valid_zip(path)]
if invalid:
    raise RuntimeError(
        'Some COCO archives are missing or invalid: ' + ', '.join(invalid) +
        '. Re-run Step 3 before extraction.'
    )

raw_markers = [
    COCO_RAW_DIR / 'train2017',
    COCO_RAW_DIR / 'val2017',
    COCO_RAW_DIR / 'annotations' / 'instances_train2017.json',
    COCO_RAW_DIR / 'annotations' / 'instances_val2017.json',
]
raw_ready = all(path.exists() for path in raw_markers)

if raw_ready and not FORCE_EXTRACT_COCO:
    print('[SKIP] COCO raw structure already exists in Drive. Extraction skipped.')
else:
    extracted = extract_coco_2017_archives(
        COCO_ZIPS_DIR,
        COCO_RAW_DIR,
        overwrite=bool(FORCE_EXTRACT_COCO),
    )
    for name, path in extracted.items():
        print(f'[OK] extracted {name}: {path}')


In [ ]:
# Step 5: convert COCO raw -> COCO-small YOLO.
from src.data.coco_small_manager import CocoSmallPrepareConfig, prepare_coco_small_yolo

FORCE_REBUILD_COCO_SMALL_YOLO = False

prepare_cfg = CocoSmallPrepareConfig(
    small_max_area=32.0**2,
    keep_images_without_small=False,
    include_crowd=False,
    splits=('train', 'val'),
    link_images=True,
)

yolo_markers = [
    COCO_SMALL_YOLO_DIR / 'images' / 'train',
    COCO_SMALL_YOLO_DIR / 'images' / 'val',
    COCO_SMALL_YOLO_DIR / 'labels' / 'train',
    COCO_SMALL_YOLO_DIR / 'labels' / 'val',
    COCO_SMALL_YOLO_DIR / 'coco_small.yaml',
]


def _count_images(images_dir: Path) -> int:
    exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    if not images_dir.exists():
        return 0
    return sum(1 for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in exts)


yolo_ready = all(p.exists() for p in yolo_markers)
train_count = _count_images(COCO_SMALL_YOLO_DIR / 'images' / 'train')
val_count = _count_images(COCO_SMALL_YOLO_DIR / 'images' / 'val')
yolo_ready = yolo_ready and (train_count > 0 and val_count > 0)

if yolo_ready and not FORCE_REBUILD_COCO_SMALL_YOLO:
    print('[SKIP] COCO-small YOLO dataset already exists in Drive.')
    print(f'[INFO] train images: {train_count}, val images: {val_count}')
else:
    prepare_report = prepare_coco_small_yolo(
        raw_coco_root=COCO_RAW_DIR,
        output_root=COCO_SMALL_YOLO_DIR,
        config=prepare_cfg,
    )
    print('is_valid:', prepare_report.is_valid)
    print('num_classes:', len(prepare_report.class_names))
    print('train stats:', prepare_report.splits.get('train', {}))
    print('val stats:', prepare_report.splits.get('val', {}))


In [ ]:
# Step 6: run analysis/policy + selected training modes + optional evaluation (Drive-direct mode).
from pathlib import Path
import shutil
import yaml
from src.pipeline_mvp import run_mvp_pipeline, _dataset_class_names, _evaluate_training_runs, _write_runtime_data_yaml
from src.training.train_runner import TrainRunConfig, run_train_mode
from src.augmentation.policy_to_ultralytics import AugmentationPolicy
from src.augmentation.object_bank import ObjectBank
from src.utils.io import load_json, load_yaml

TRAIN_MODES = ['baseline', 'manual', 'adaptive']  # choose any subset
RUN_EVAL = True
USE_LOCAL_RUNTIME_DATASET = False
LOCAL_DATASET_ROOT = Path('/content/datasets/coco_small_yolo')
TRAIN_PROFILE = 'hour'
TRAIN_EPOCHS = 25
TRAIN_IMGSZ = 960
TRAIN_BATCH = 6
TRAIN_WORKERS = 1
TRAIN_FRACTION = 1.0
TRAIN_PATIENCE = 12
SYNC_ARTIFACTS_TO_DRIVE = True

valid_modes = {'baseline', 'manual', 'adaptive'}
invalid_modes = [m for m in TRAIN_MODES if m not in valid_modes]
assert not invalid_modes, f'Unsupported TRAIN_MODES: {invalid_modes}'
assert TRAIN_MODES, 'TRAIN_MODES must not be empty.'
print('[INFO] Drive-direct mode: training uses dataset directly from Google Drive (no local copy).')

if USE_LOCAL_RUNTIME_DATASET:
    train_marker = LOCAL_DATASET_ROOT / 'images' / 'train'
    val_marker = LOCAL_DATASET_ROOT / 'images' / 'val'
    if not (train_marker.exists() and val_marker.exists()):
        print(f'[INFO] Copy dataset to local runtime: {COCO_SMALL_YOLO_DIR} -> {LOCAL_DATASET_ROOT}')
        if LOCAL_DATASET_ROOT.exists():
            shutil.rmtree(LOCAL_DATASET_ROOT)
        LOCAL_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(COCO_SMALL_YOLO_DIR, LOCAL_DATASET_ROOT)
    else:
        print(f'[OK] Reuse local dataset: {LOCAL_DATASET_ROOT}')
    runtime_dataset_root = LOCAL_DATASET_ROOT
else:
    runtime_dataset_root = COCO_SMALL_YOLO_DIR

base_cfg_path = PROJECT_ROOT / 'configs' / 'coco_small_config.yaml'
runtime_cfg_path = PROJECT_ROOT / 'artifacts' / 'coco_small_runtime_config.yaml'
runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)

with base_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['mode'] = 'manual'
cfg['dataset']['root'] = str(runtime_dataset_root)
cfg['dataset']['raw_root'] = str(COCO_RAW_DIR)
cfg['training']['data_yaml'] = str(runtime_dataset_root / 'coco_small.yaml')
cfg['training']['project_dir'] = str(COCO_EXPERIMENT_RUNS)
cfg['training']['run_ablation'] = False
cfg['training']['cache'] = False
cfg['training']['val_during_train'] = True
cfg['training']['patience'] = TRAIN_PATIENCE

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

prep_outputs = run_mvp_pipeline(
    project_config_path=runtime_cfg_path,
    run_training=False,
    run_eval=False,
    train_profile=TRAIN_PROFILE,
    auto_predict_for_eval=False,
)
print('[OK] analysis/policy completed')
print(prep_outputs)

baseline_yaml_path = PROJECT_ROOT / 'configs' / 'baseline.yaml'
manual_yaml_path = PROJECT_ROOT / 'configs' / 'manual.yaml'
adaptive_policy_json = PROJECT_ROOT / 'artifacts' / 'policy' / 'policy_adaptive.json'
object_bank_path = PROJECT_ROOT / 'artifacts' / 'policy' / 'object_bank.json'

assert baseline_yaml_path.exists(), f'Missing {baseline_yaml_path}'
assert manual_yaml_path.exists(), f'Missing {manual_yaml_path}'
assert adaptive_policy_json.exists(), f'Missing adaptive policy: {adaptive_policy_json}'

train_cfg = TrainRunConfig(
    data_yaml=str(runtime_dataset_root / 'coco_small.yaml'),
    model=str(cfg['training'].get('model', 'yolo11s.pt')),
    epochs=TRAIN_EPOCHS,
    imgsz=TRAIN_IMGSZ,
    batch=TRAIN_BATCH,
    device=cfg['training'].get('device', 0),
    workers=TRAIN_WORKERS,
    fraction=TRAIN_FRACTION,
    project_dir=str(COCO_EXPERIMENT_RUNS),
    seed=int(cfg['project'].get('seed', 42)),
    deterministic=bool(cfg['project'].get('deterministic', True)),
    rect=bool(cfg['training'].get('rect', False)),
    multi_scale=bool(cfg['training'].get('multi_scale', False)),
    baseline_disable_albumentations=bool(cfg['training'].get('baseline_disable_albumentations', True)),
    require_custom_augmentations=bool(cfg['training'].get('require_custom_augmentations', True)),
    val=bool(cfg['training'].get('val_during_train', True)),
    cache=cfg['training'].get('cache', False),
    patience=TRAIN_PATIENCE,
)

baseline_args = load_yaml(baseline_yaml_path)
manual_args = load_yaml(manual_yaml_path)
adaptive_policy = AugmentationPolicy(payload=load_json(adaptive_policy_json))
object_bank = ObjectBank.load(object_bank_path, seed=train_cfg.seed) if object_bank_path.exists() else None

run_dirs = {}
for mode in TRAIN_MODES:
    if mode == 'baseline':
        mode_args = baseline_args
        custom_aug = None
    elif mode == 'manual':
        mode_args = manual_args
        custom_aug = None
    else:
        mode_args = adaptive_policy.get_ultralytics_train_args()
        custom_aug = adaptive_policy.get_albumentations_transforms(object_bank=object_bank, seed=train_cfg.seed)

    run_dir = run_train_mode(
        mode=mode,
        config=train_cfg,
        mode_args=mode_args,
        custom_augmentations=custom_aug,
    )
    run_dirs[mode] = str(run_dir)

print('[OK] Training completed for modes:', TRAIN_MODES)
print(run_dirs)

if RUN_EVAL:
    class_names = _dataset_class_names(cfg=cfg, dataset_root=runtime_dataset_root)
    image_extensions = tuple(cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp']))
    runtime_data_yaml = _write_runtime_data_yaml(runtime_dataset_root, class_names)
    cfg['training']['data_yaml'] = str(runtime_data_yaml)
    metrics_by_run, pred_dirs = _evaluate_training_runs(
        cfg=cfg,
        dataset_root=runtime_dataset_root,
        image_extensions=image_extensions,
        class_names=class_names,
        run_names=list(TRAIN_MODES),
    )
    print('[OK] Evaluation completed')
    print(metrics_by_run)
    print(pred_dirs)

if SYNC_ARTIFACTS_TO_DRIVE:
    local_artifacts = PROJECT_ROOT / 'artifacts'
    COCO_EXPERIMENT_ARTIFACTS.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_artifacts, COCO_EXPERIMENT_ARTIFACTS, dirs_exist_ok=True)
    print('[OK] Artifacts synced to Drive:', COCO_EXPERIMENT_ARTIFACTS)

